# v-star: Risk Analysis with Go + Python

This notebook demonstrates calling the high-performance v-star Go engine from Python.

**What it shows:**
- Monte Carlo interest rate simulation (100k paths in ~1 second)
- Value at Risk (VaR) and Conditional Tail Expectation (CTE)
- Visualization with matplotlib

In [ ]:
import subprocess
import re
import matplotlib.pyplot as plt
import numpy as np

## Build v-star (skip if already built)

In [ ]:
import os
os.chdir('../..')  # Go to repo root
result = subprocess.run(['go', 'build', '-o', 'v-star.exe', './cmd/v-star'], capture_output=True, text=True)
if result.returncode != 0:
    print(f'Build failed: {result.stderr}')
else:
    print('v-star built successfully')

## Run Monte Carlo Simulation

Simulate 100,000 interest rate paths using geometric Brownian motion.
Parameters: initial rate 5%, drift 2%, volatility 15%, 10 annual steps.

In [ ]:
result = subprocess.run(
    ['./v-star.exe', 'montecarlo', '--paths=100000', '--steps=10', '--drift=0.02', '--volatility=0.15', '--seed=42'],
    capture_output=True, text=True
)
print(result.stdout)

## Parse Results and Visualize

In [ ]:
# Re-run and capture the final rate percentiles for visualization
result = subprocess.run(
    ['./v-star.exe', 'montecarlo', '--paths=100000', '--steps=10', '--drift=0.02', '--volatility=0.15', '--seed=42'],
    capture_output=True, text=True
)

# Extract percentiles from output
lines = result.stdout.split('\n')
percentiles = {}
for line in lines:
    match = re.search(r'([\dth]+\s*\(.*?\)|[\dth]+):\s*([\d.]+)%', line)
    if match:
        name = match.group(1).strip()
        value = float(match.group(2))
        percentiles[name] = value

print('Parsed percentiles:', percentiles)

In [ ]:
# Simulate rate distribution for histogram
np.random.seed(42)
n_paths = 100000
steps = 10
initial_rate = 0.05
mu = 0.02
sigma = 0.15
dt = 1.0

# Generate paths using GBM (same algorithm as v-star)
rates = np.full(n_paths, initial_rate)
for _ in range(steps):
    z = np.random.standard_normal(n_paths)
    drift = (mu - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * z
    rates = rates * np.exp(drift + diffusion)

# Plot histogram of final rates
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(rates * 100, bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(np.percentile(rates, 5) * 100, color='red', linestyle='--', label='VaR 95%')
axes[0].axvline(np.percentile(rates, 1) * 100, color='darkred', linestyle='--', label='VaR 99%')
axes[0].set_xlabel('Final Interest Rate (%)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Final Rates (100k Monte Carlo Paths)')
axes[0].legend()

# Fan chart - sample paths
sample_paths = 50
path_rates = np.full((sample_paths, steps + 1), initial_rate)
for t in range(1, steps + 1):
    z = np.random.standard_normal(sample_paths)
    drift = (mu - 0.5 * sigma**2) * dt
    diffusion = sigma * np.sqrt(dt) * z
    path_rates[:, t] = path_rates[:, t-1] * np.exp(drift + diffusion)

for i in range(sample_paths):
    axes[1].plot(range(steps + 1), path_rates[i] * 100, alpha=0.15, color='steelblue')

# Add percentile bands
p5 = np.percentile(path_rates, 5, axis=0) * 100
p50 = np.percentile(path_rates, 50, axis=0) * 100
p95 = np.percentile(path_rates, 95, axis=0) * 100
axes[1].plot(range(steps + 1), p50, color='red', linewidth=2, label='Median')
axes[1].fill_between(range(steps + 1), p5, p95, alpha=0.3, color='orange', label='90% CI')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Interest Rate (%)')
axes[1].set_title('Monte Carlo Rate Paths (Geometric Brownian Motion)')
axes[1].legend()

plt.tight_layout()
plt.savefig('monte_carlo_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: monte_carlo_analysis.png')

## Key Takeaways

- v-star generates 100k Monte Carlo paths in ~1 second from Go
- Python handles visualization and analysis
- This Go+Python integration pattern works for any quantitative pipeline

**Why this matters for actuarial/finance roles:**
- Demonstrates Monte Carlo simulation (core quant skill)
- Shows VaR/CTE computation (risk management)
- High-performance Go engine + Python analysis = production-ready pattern